In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "Milos/slovak-gpt-j-405M"

print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# GPT-J does not have a pad token by default
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Pipeline approach
# text_gen_pipeline = pipeline(
#     "text-generation",
#     model=model_name,
#     tokenizer=model_name,
#     device=0 if torch.cuda.is_available() else -1
# )

# print("\n--- Text Generation Example (Pipeline Way) ---")

# prompt_pipeline_1 = "Vláda dnes oznámila nové opatrenia"
# result_pipeline_1 = text_gen_pipeline(prompt_pipeline_1, max_new_tokens=80, do_sample=True, temperature=0.9)
# print(f"\nPrompt: '{prompt_pipeline_1}'")
# print(f"Generated: {result_pipeline_1[0]['generated_text']}")

# prompt_pipeline_2 = "Slovenská ekonomika v tomto roku"
# result_pipeline_2 = text_gen_pipeline(prompt_pipeline_2, max_new_tokens=80, do_sample=True, temperature=0.9)
# print(f"\nPrompt: '{prompt_pipeline_2}'")
# print(f"Generated: {result_pipeline_2[0]['generated_text']}")

print("\n--- Manual Text Generation Example ---")

def generate_text_manual(
    prompt,
    model,
    tokenizer,
    device,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.9,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.1,
):
    # 1. Tokenize the prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=True,
        truncation=True,
        max_length=512,
    ).to(device)

    prompt_length = inputs["input_ids"].shape[1]

    # 2. Generate tokens
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # 3. Decode only newly generated tokens
    generated_ids = output_ids[0][prompt_length:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return {
        "prompt": prompt,
        "generated": generated_text,
        "full_text": prompt + generated_text,
    }

# Example 1: News-article style prompt (lukasjanek/slovaksum domain)
prompt_1 = "Vláda dnes oznámila nové opatrenia v oblasti"
result_1 = generate_text_manual(prompt_1, model, tokenizer, device)
print(f"\nPrompt: '{result_1['prompt']}'")
print(f"Generated continuation: '{result_1['generated']}'")
print(f"Full text: '{result_1['full_text']}'")

# Example 2: Economic/current-events style prompt
prompt_2 = "Slovenská ekonomika v tomto roku zaznamenala"
result_2 = generate_text_manual(prompt_2, model, tokenizer, device)
print(f"\nPrompt: '{result_2['prompt']}'")
print(f"Generated continuation: '{result_2['generated']}'")
print(f"Full text: '{result_2['full_text']}'")

Loading Milos/slovak-gpt-j-405M...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/836 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/815M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

GPTJForCausalLM LOAD REPORT from: Milos/slovak-gpt-j-405M
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...23}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/814M [00:00<?, ?B/s]


--- Manual Text Generation Example ---

Prompt: 'Vláda dnes oznámila nové opatrenia v oblasti'
Generated continuation: ' verejného obstarávania

BRATISLAVA. Zabrániť stratám na verejných zákazkách z dôvodu neférových a netransparentných postupov pri obstarávaní tovarov, služieb alebo stavebných prác bude možné podľa zámeru Ministerstva hospodárstva (MH) SR prostredníctvom novely zákona o verejnom obstarávaní. Vyplýva to zo zverejneného návrhu legislatívneho uznesenia vlády k novele zákona č. 25 / 2006 Z. z. o verejnom obstarávaní, ktorým sa má'
Full text: 'Vláda dnes oznámila nové opatrenia v oblasti verejného obstarávania

BRATISLAVA. Zabrániť stratám na verejných zákazkách z dôvodu neférových a netransparentných postupov pri obstarávaní tovarov, služieb alebo stavebných prác bude možné podľa zámeru Ministerstva hospodárstva (MH) SR prostredníctvom novely zákona o verejnom obstarávaní. Vyplýva to zo zverejneného návrhu legislatívneho uznesenia vlády k novele zákona č. 25 / 2006 Z. z.

In [ ]:
!rm -rf ~/.cache/huggingface
print("Hugging Face cache cleared.")

Hugging Face cache cleared.
